In [12]:
import pandas as pd
import re
import numpy as np
from deep_translator import GoogleTranslator

In [13]:
pd.set_option('display.max_colwidth', None)

In [14]:
rassereno_path = '/Users/emma/projects/cognoscolab/rassereno'
math_ma_llm = '/Users/emma/projects/cognoscolab/math_anxiety_LLMs'

## MAS-IT dataframes merge

In [15]:
mas_it_2023 = pd.read_csv(f'{math_ma_llm}/out/formatting/humans_2023/mas_it_2023_humans.csv')
mas_it_humans_2024 = pd.read_csv(f'{math_ma_llm}/out/formatting/humans_2024/mas_it_2024_humans.csv')
mas_it_BA_2024 = pd.read_csv(f'{rassereno_path}/out/formatting/psychology_BA_2024/mas_it_2024.csv')
mas_it_2025 = pd.read_csv(f'{rassereno_path}/out/formatting/all_students_2025/mas_it_2025.csv')

In [16]:
# gets all the physics students as saves the df to csv
mas_it_fisica = mas_it_2025[mas_it_2025['participant_id'].str.startswith('fisica')].reset_index(drop=True)

mas_it_fisica.to_csv(f'{rassereno_path}/out/formatting/physics_2025/mas_it_fisica_2025.csv', index=False)
print('Number of physics students:', len(mas_it_fisica))

Number of physics students: 10


In [17]:
# gets all the high school students as saves the df to csv
mas_it_liceo_2025 = mas_it_2025[mas_it_2025['participant_id'].str.startswith('liceo')].reset_index(drop=True)

mas_it_liceo_2025.to_csv(f'{rassereno_path}/out/formatting/liceo_2025/mas_it_liceo_2025.csv', index=False)
print('Number of high school students:', len(mas_it_liceo_2025))

Number of high school students: 62


In [18]:
# gets all the university students of 2025 and saves it in a csv file
mas_it_psychology_2025 = mas_it_2025[mas_it_2025['participant_id'].str.startswith('interfacce')].reset_index(drop=True)
print('Number of psychology students of 2025:', len(mas_it_psychology_2025))

Number of psychology students of 2025: 23


In [19]:
# get the column names of the 2025 dataset
cols_2025 = mas_it_psychology_2025.columns

# restrict all dataframes to those columns before concatenation
dfs = [
    mas_it_humans_2024[cols_2025.intersection(mas_it_humans_2024.columns)],
    mas_it_2023[cols_2025.intersection(mas_it_2023.columns)],
    mas_it_BA_2024[cols_2025.intersection(mas_it_BA_2024.columns)],
    mas_it_psychology_2025[cols_2025]
]

# concatenate them
all_psycology_mas_it = pd.concat(dfs, ignore_index=True)
all_psycology_mas_it.to_csv(f'{rassereno_path}/out/formatting/all_psychology_students/mas_it_all_psy_students.csv', index=False)

print('Number of all the psychology students:', len(all_psycology_mas_it))

Number of all the psychology students: 316


## Associations and Valence dataframes merge

In [20]:
associations_valence_2023 = pd.read_csv(f'{math_ma_llm}/out/formatting/humans_2023/associations_valence_2023_humans.csv')
associations_valence_2024 = pd.read_csv(f'{math_ma_llm}/out/formatting/humans_2024/associations_valence_2024_humans.csv')
associations_valence_BA_2024 = pd.read_csv(f'{rassereno_path}/out/formatting/psychology_BA_2024/associations_valence_2024.csv')
associations_valence_2025 = pd.read_csv(f'{rassereno_path}/out/formatting/all_students_2025/associations_valence_task_2025.csv')
associations_valence_gpt_oss = pd.read_csv(f'{rassereno_path}/out/formatting/gpt_oss/associations_valence_physics.csv')

In [21]:
associations_valence_psychology_2025 = associations_valence_2025[associations_valence_2025['participant_id'].str.startswith('interfacce')]

associations_valence_fisica = associations_valence_2025[associations_valence_2025['participant_id'].str.startswith('fisica')].reset_index(drop=True)

associations_valence_fisica.to_csv(f'{rassereno_path}/out/formatting/physics_2025/associations_valence_fisica.csv', index=False)

associations_valence_liceo_2025 = associations_valence_2025[associations_valence_2025['participant_id'].str.startswith('liceo')]

associations_valence_liceo_2025.to_csv(f'{rassereno_path}/out/formatting/liceo_2025/associations_valence_liceo_2025.csv', index=False)

In [22]:
all_psychology_associations_valence = pd.concat([associations_valence_2024, associations_valence_2023, associations_valence_BA_2024, associations_valence_psychology_2025], ignore_index=True)

all_psychology_associations_valence.to_csv(f'{rassereno_path}/out/formatting/all_psychology_students/associations_valence_all_psy_students.csv', index=False)

## Mergign whole human and GPT datasets

### Human participants

In [23]:
def participant_id_mapping(mas_it_df = pd.DataFrame(), associations_valence_df = pd.DataFrame(), sample = str):
    mas_it_copy = mas_it_df.copy()
    associations_valence_copy = associations_valence_df.copy()

    for i in range(len(mas_it_copy)):
        mas_it_copy.loc[i, 'participant_id'] = f'{sample}_{i+1:03}'

    map_participant_id = {mas_it_df.loc[i, 'participant_id'] : mas_it_copy.loc[i, 'participant_id'] for i in range(len(mas_it_copy))}

    associations_valence_copy['participant_id'] = associations_valence_df['participant_id'].map(map_participant_id)

    return mas_it_copy, associations_valence_copy

In [24]:
psy_mas_it, psy_asociations_valence = participant_id_mapping(all_psycology_mas_it, all_psychology_associations_valence, 'psychology')

high_school_mas_it, high_school_associations_valence = participant_id_mapping(mas_it_liceo_2025, associations_valence_liceo_2025, 'high_school')

physics_mas_it, physics_associations_valence = participant_id_mapping(mas_it_fisica, associations_valence_fisica, 'physics')

In [25]:
all_humans_associations_valence = pd.concat([psy_asociations_valence, physics_associations_valence, high_school_associations_valence], axis=0).reset_index(drop=True)
all_humans_associations_valence.to_csv(f'{rassereno_path}/out/formatting/database/all_humans_associations_valence.csv', index=False)
all_humans_mas_it = pd.concat([psy_mas_it, physics_mas_it, high_school_mas_it], axis = 0).reset_index(drop=True)
all_humans_mas_it.to_csv(f'{rassereno_path}/out/formatting/database/all_humans_mas_it.csv', index=False)

### GPT-oss participants

In [26]:
associations_gpt_psychology = pd.read_csv(f'{rassereno_path}/out/formatting/gpt_oss/associations_valence_psychology.csv')
mas_it_gpt_psychology = pd.read_csv(f'{rassereno_path}/out/formatting/gpt_oss/mas_it_psychology.csv')

associations_gpt_physics = pd.read_csv(f'{rassereno_path}/out/formatting/gpt_oss/associations_valence_physics.csv')
mas_it_gpt_physics = pd.read_csv(f'{rassereno_path}/out/formatting/gpt_oss/mas_it_physics.csv')

associations_gpt_liceo = pd.read_csv(f'{rassereno_path}/out/formatting/gpt_oss/associations_valence_liceo.csv')
mas_it_gpt_liceo = pd.read_csv(f'{rassereno_path}/out/formatting/gpt_oss/mas_it_liceo.csv')

gpt_psychology_mas_it, gpt_psychology_associations_valence = participant_id_mapping(mas_it_gpt_psychology, associations_gpt_psychology, 'gpt_oss_psychology')

gpt_physics_mas_it, gpt_physics_associations_valence = participant_id_mapping(mas_it_gpt_physics, associations_gpt_physics, 'gpt_oss_physics')

gpt_high_school_mas_it, gpt_high_school_associations = participant_id_mapping(mas_it_gpt_liceo, associations_gpt_liceo, 'gpt_oss_high_school')

In [27]:
all_gpt_associations_valence = pd.concat([gpt_psychology_associations_valence, gpt_physics_associations_valence, gpt_high_school_associations], axis=0).reset_index(drop=True)
all_gpt_associations_valence.to_csv(f'{rassereno_path}/out/formatting/database/all_gpt_associations_valence.csv', index=False)
all_gpt_mas_it = pd.concat([gpt_psychology_mas_it, gpt_physics_mas_it, gpt_high_school_mas_it], axis = 0).reset_index(drop=True)
all_gpt_mas_it.to_csv(f'{rassereno_path}/out/formatting/database/all_gpt_mas_it.csv', index=False)

## Common words of multiple datasets

In [28]:
common_words_university = list(set(associations_valence_2024['cue_word']) & set(associations_valence_2023['cue_word']) & set(associations_valence_BA_2024['cue_word']) & set(all_psychology_associations_valence
['cue_word']))

print(f'Common word of all university subgroups: {common_words_university}')

Common word of all university subgroups: ['ansia', 'esame', 'equazione', 'sfida', 'curiosità', 'fisica', 'matematica', 'biologia', 'insegnante', 'statistica']


In [29]:
set(associations_valence_gpt_oss['cue_word']) - set(associations_valence_2024['cue_word'])

{'creativita', 'curiosita', 'personalita', 'universita'}

In [30]:
word_to_category = {
    'algoritmo': 'STEM Disciplines',
    'analisi': 'STEM Disciplines',
    'biologia': 'STEM Disciplines',
    'calcolo': 'STEM Disciplines',
    'dimostrazione': 'STEM Disciplines',
    'equazione': 'STEM Disciplines',
    'fisica': 'STEM Disciplines',
    'informatica': 'STEM Disciplines',
    'ingegneria': 'STEM Disciplines',
    'matematica': 'STEM Disciplines',
    'matematico/a': 'STEM Disciplines',
    'numeri': 'STEM Disciplines',
    'performance_in_matematica': 'STEM Disciplines',
    'statistica': 'STEM Disciplines',
    'tecnologia': 'STEM Disciplines',
    'test_di_matematica': 'STEM Disciplines',
    'utilità_della_matematica': 'STEM Disciplines',
    'dati': 'STEM Disciplines',
    'grafico': 'STEM Disciplines',
    'ipotesi': 'STEM Disciplines',
    'laboratorio': 'STEM Disciplines',
    'logica': 'STEM Disciplines',
    'ricerca': 'STEM Disciplines',
    'scienza': 'STEM Disciplines',
    'scienziata': 'STEM Disciplines',
    'scienziato': 'STEM Disciplines',
    'scoperta': 'STEM Disciplines',
    'esperimento': 'STEM Disciplines',
    'innovazione': 'STEM Disciplines',
    'neuroscienze': 'STEM Disciplines',

    'classe': 'Academic Environment',
    'compagni': 'Academic Environment',
    'docente': 'Academic Environment',
    'genitore': 'Academic Environment',
    'genitori': 'Academic Environment',
    'insegnante': 'Academic Environment',
    'studente': 'Academic Environment',
    'studente/ssa': 'Academic Environment',
    'professore': 'Academic Environment',
    'lavagna': 'Academic Environment',
    'mappa_concettuale': 'Academic Environment',
    'note': 'Academic Environment',
    'scuola': 'Academic Environment',
    'università': 'Academic Environment',

    'esame': 'Evaluation',
    'esercitazione': 'Evaluation',
    'prova': 'Evaluation',
    'test': 'Evaluation',
    'media': 'Evaluation',
    'valutazione': 'Evaluation',
    'voto': 'Evaluation',

    'cambio_di_piani': 'Learning and Motivation',
    'compito': 'Learning and Motivation',
    'concentrazione': 'Learning and Motivation',
    'confusione': 'Learning and Motivation',
    'conoscenza': 'Learning and Motivation',
    'creatività': 'Learning and Motivation',
    'curiosità': 'Learning and Motivation',
    'distrazione': 'Learning and Motivation',
    'divertimento': 'Learning and Motivation',
    'errori': 'Learning and Motivation',
    'fissare_un_obiettivo': 'Learning and Motivation',
    'impegno': 'Learning and Motivation',
    'importanza': 'Learning and Motivation',
    'intelligenza': 'Learning and Motivation',
    'memorizzare': 'Learning and Motivation',
    'motivazione': 'Learning and Motivation',
    'pianificare': 'Learning and Motivation',
    'problema': 'Learning and Motivation',
    'programma': 'Learning and Motivation',
    'programmare': 'Learning and Motivation',
    'programmazione': 'Learning and Motivation',
    'revisionare': 'Learning and Motivation',
    'rimandare': 'Learning and Motivation',
    'scadenza': 'Learning and Motivation',
    'sfida': 'Learning and Motivation',
    'sforzo': 'Learning and Motivation',
    'soluzione': 'Learning and Motivation',
    'studiare': 'Learning and Motivation',
    'successo': 'Learning and Motivation',
    'tenere_traccia': 'Learning and Motivation',
    'futuro': 'Learning and Motivation',
    'modello': 'Learning and Motivation',
    'sapere': 'Learning and Motivation',
    'regole': 'Learning and Motivation',
    'relazione': 'Learning and Motivation',
    'sviluppo': 'Learning and Motivation',
    'passione': 'Learning and Motivation',
    
    'ansia': 'Mental Health',
    'arte': 'Mental Health',
    'attaccamento': 'Mental Health',
    'benessere': 'Mental Health',
    'cognizione': 'Mental Health',
    'counselling': 'Mental Health',
    'emozione': 'Mental Health',
    'fallimento': 'Mental Health',
    'frustrazione': 'Mental Health',
    'imbarazzo': 'Mental Health',
    'meditazione': 'Mental Health',
    'mente': 'Mental Health',
    'noia': 'Mental Health',
    'panico': 'Mental Health',
    'paura': 'Mental Health',
    'personalità': 'Mental Health',
    'psicologia': 'Mental Health',
    'psicopatologia': 'Mental Health',
    'psicoterapia': 'Mental Health',
    'terapeuta': 'Mental Health',
    'terapia': 'Mental Health',
    'stress': 'Mental Health'
}

In [31]:
def cue_words_summary(df):
    n_cue_words = len(set(df['cue_word']))
    n_participants = len(set(df['participant_id']))

    words = sorted(df['cue_word'].unique())
    categories = ['STEM Disciplines', 'Academic Environment', 'Evaluation', 'Learning and Motivation', 'Mental Health']
    grouped_words = {cat: [w for w in words if word_to_category.get(w) == cat] for cat in categories}
    
    translated_grouped_words = {}
    for cat, words_list in grouped_words.items():
        translated_grouped_words[cat] = [
            GoogleTranslator(source='it', target='en').translate(word.capitalize())
            for word in words_list
        ]

    df = pd.DataFrame(dict([(k, pd.Series(v)) for k,v in translated_grouped_words.items()]))
    print(f"len df {len(df)}; {n_cue_words} words for {n_participants} participants")
    df.insert(0, 'N', np.repeat('', len(df)))
    
    return df

df_2023 = cue_words_summary(associations_valence_2023)
df_2024 = cue_words_summary(associations_valence_2024)
df_BA_2024 = cue_words_summary(associations_valence_BA_2024)
df_2025 = cue_words_summary(associations_valence_psychology_2025)

df_summary = pd.concat([df_2023, df_2024, df_BA_2024, df_2025], ignore_index=True)

len df 26; 51 words for 57 participants
len df 11; 40 words for 70 participants
len df 11; 41 words for 166 participants
len df 18; 42 words for 23 participants


In [37]:
df_summary.to_csv(f'{rassereno_path}/out/formatting/cue_words_summary.csv', index=False)

In [36]:
latex = df_summary.to_latex(multirow=True, position='!hbpt',
                           caption='Cue words for different samples.',
                           label='tab:cue_words_summary',
                           index=False).replace('_', ' ').replace('NaN', '')

with open(f'{rassereno_path}/out/formatting/cue_words_summary_latex.txt', 'w') as f:
    f.write(latex)

In [33]:
gpt_oss_associations_valence = pd.read_csv(f'{rassereno_path}/out/formatting/associations_valence_gpt_oss_physics.csv')

FileNotFoundError: [Errno 2] No such file or directory: '/Users/emma/projects/cognoscolab/rassereno/out/formatting/associations_valence_gpt_oss_physics.csv'

In [ ]:
gpt_oss_words = cue_words_summary(gpt_oss_associations_valence, [])

In [ ]:
list(set(associations_valence_liceo_2025['cue_word']) & set(all_psychology_associations_valence['cue_word']) & set(gpt_oss_associations_valence['cue_word']))

['benessere',
 'equazione',
 'esame',
 'sfida',
 'statistica',
 'matematica',
 'scienziato',
 'innovazione',
 'informatica',
 'ansia',
 'scienza',
 'fisica']

In [ ]:
def cue_word_extraction(file):
    data = open(file).readlines()

    pairs = [re.findall(r'"([^"]+)"', line) for line in data]

    df = pd.DataFrame(pairs, columns=['association_0', 'association_1'])

    counts = df['association_0'].value_counts()

    cue_words = counts[:len(counts) // 2]

    cue_words_list = cue_words.index.tolist()

    return df, cue_words_list

In [ ]:
experts_df_unfiltered, cue_words_experts = cue_word_extraction(f'{rassereno_path}/data/data_2019/Experts_unfiltered.txt')

In [ ]:
cue_words_experts

['science',
 'university',
 'system',
 'mathematics',
 'physics',
 'art',
 'school',
 'chemistry',
 'life',
 'complex',
 'conference',
 'network',
 'market',
 'analysis',
 'loop',
 'biology',
 'electricity',
 'solid',
 'language',
 'brain',
 'statistics',
 'word',
 'ants',
 'chaos',
 'cloud',
 'scientific',
 'theory',
 'studies',
 'draw',
 'dna',
 'degree',
 'evolution',
 'metal',
 'integral',
 'failure',
 'individual',
 'derivative',
 'element',
 'teaching',
 'model',
 'transistor',
 'force',
 'hub',
 'attractor',
 'theorem',
 'liquid',
 'flow',
 'geography',
 'reasoning',
 'prisoner',
 'geometry',
 'mathematician',
 'character',
 'spike',
 'water',
 'period',
 'health',
 'teacher',
 'instrument',
 'probability',
 'cell',
 'matrix',
 'state',
 'magnetism',
 'approximation',
 'island',
 'curricula',
 'link',
 'cycle',
 'tool',
 'computer',
 'suggestion',
 'saddle',
 'metric',
 'discovery',
 'biosphere',
 'mark',
 'place',
 'emergence',
 'charge',
 'star',
 'flower',
 'town',
 'heat',
 

In [ ]:
students_df_unfiltered, cue_words_students_unfiltered = cue_word_extraction(f'{rassereno_path}/data/data_2019/Students_unFiltered.txt')

In [ ]:
students_df_filtered, cue_words_students_filtered = cue_word_extraction(f'{rassereno_path}/data/data_2019/Students_Filtered.txt')

In [ ]:
list(set(associations_valence_liceo_2025['cue_word']) & set(all_psychology_associations_valence['cue_word']) & set(gpt_oss_associations_valence['cue_word']) & set(cue_words_students_unfiltered))

['matematica', 'scienziato', 'fisica', 'esame']

## Experts and high school 2019 data

### Check if valences are for filtered or unfiltered df

In [ ]:
# counts how many valence ratings per word exist in ValenceStudentsUPD.txt
counts = {}
with open(f"{rassereno_path}/data/data_2019/ValenceStudentsUPD.txt", "r") as f:
    for line in f:
        parts = line.strip().split()
        # gets the first element of each row, which is the word
        word = parts[0]
        # gets all the numbers after the word, removing the .
        numbers = [p for p in parts[1:] if p.replace('.', '').isdigit()]
        # counts how many valence ratings per word exist
        counts[word] = counts.get(word, 0) + len(numbers)
df_counts = pd.DataFrame(list(counts.items()), columns=['word', 'count_valence'], index=None)

In [ ]:
def count_students_valence(df, file_name):
    combined_counts = (
        df['association_0'].value_counts()
        .add(df['association_1'].value_counts(), fill_value=0)
        .astype(int)
    )

    combined_counts = pd.DataFrame(list(combined_counts.items()), columns=['word', 'count_cue_ass_word'], index=None)

    df = pd.merge(df_counts, combined_counts, on='word', how='outer')
    df.to_csv(f'{rassereno_path}/out/formatting/students_2019/combined_counts_{file_name}.csv', index=False)

In [ ]:
count_students_valence(students_df_unfiltered, 'unfiltered')
count_students_valence(students_df_filtered, 'filtered')

### Creation df for word and associations

In [ ]:
# Assign valences sequentially
def assign_valence(row, col_name, valence_map):
    word = row[col_name]
    if word in valence_map and len(valence_map[word]) > 0:
        return float(valence_map[word].pop(0).replace('.', ''))
    else:
        return np.nan

In [ ]:
def association_valence_df(input, df, output):
    valences = open(f"{rassereno_path}/data/data_2019/{input}.txt", "r").readlines()
    valences = [v.strip().split('\t') for v in valences]

    valence_map = {}
    for row in valences:
        if not row[0].replace('.', '').isdigit():
            word = row[0]
            vals = [v for v in row[1:] if v.replace('.', '').isdigit()]
            valence_map.setdefault(word, []).extend(vals)

    df['valence_association_1'] = df.apply(assign_valence, axis=1, args=(('association_1', valence_map)))
    df['valence_association_0'] = df.apply(assign_valence, axis=1, args=(('association_0', valence_map)))

    # df = df['association_0', 'valence_association_0', 'association_1', 'valence_association_1']

    df['new_index'] = df.index + 1
    df['cue_word_original'] = df['association_0']

    # melt from wide to long
    df_long = pd.wide_to_long(df, stubnames=['association', 'valence_association'], i=['new_index', 'cue_word_original'], j='index', sep='_', suffix=r'\d+').reset_index()

    # optional: rename columns for clarity
    df_long = df_long.rename(columns={
        'association': 'word',
        'valence_association': 'valence', 
        'cue_word_original': 'cue_word'
    })

    columns_to_clean = ['cue_word', 'word']
    for col in columns_to_clean:
        if col in df_long.columns:
            df_long[col] = (
                df_long[col]
                .astype(str)
                .str.lower()
                .str.replace(r'[\d\W_]+', ' ', regex=True)
                .str.strip()
            )
    df_long = df_long.dropna()

    df_long = df_long[(df_long['word'] != 'nan') & (df_long['word'] != '')]
    df_long.to_csv(f'{rassereno_path}/out/formatting/{output}/associations_valence_{output}.csv', index=False)

    return df_long

In [ ]:
students_ass_val_df = association_valence_df('ValenceStudentsUPD', students_df_unfiltered, 'students_2019')

In [ ]:
experts_ass_val_df = association_valence_df('ValenceExperts', experts_df_unfiltered, 'experts_2019')

## Txt file writing

In [ ]:
with open(f"{rassereno_path}/out/formatting/participants_rassereno.txt", "a") as f:
  f.write("Number of participants for Rassereno article:\n"
  "Experts 2019: 59\n"
  "High school students 2019: 159\n"
  f"High school students 2025: {len(mas_it_liceo_2025)}\n"
  f"Physics students 2025: {len(mas_it_fisica)}\n"
  f"All psychology students: {len(all_psycology_mas_it)} (2025 psy students: {len(mas_it_psychology_2025)})")